# ContentOps: NumPy, Pandas и Matplotlib

Воспроизводимая демонстрация data stack на безопасном синтетическом наборе ContentOps. Все экспортируемые результаты размещаются в игнорируемом `artifacts/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registers the 3D projection

from app.analytics.data import load_content_frame

ARTIFACTS = Path('../artifacts') if Path('../app').exists() else Path('artifacts')
ARTIFACTS.mkdir(exist_ok=True)
rng = np.random.default_rng(42)

## NumPy: генерация, маскирование, broadcasting, linalg и файлы

In [ ]:
matrix = rng.normal(loc=10, scale=2, size=(4, 3))
mask = matrix[:, 0] > 10
selected_rows = matrix[mask]                 # boolean masking
centered = matrix - matrix.mean(axis=0)       # broadcasting по строкам
gram = centered.T @ centered
eigenvalues, eigenvectors = np.linalg.eigh(gram)
np.save(ARTIFACTS / 'notebook_matrix.npy', matrix)
np.savetxt(ARTIFACTS / 'notebook_matrix.txt', matrix, fmt='%.4f')
loaded_npy = np.load(ARTIFACTS / 'notebook_matrix.npy')
loaded_txt = np.loadtxt(ARTIFACTS / 'notebook_matrix.txt')
print(selected_rows.shape, eigenvalues.round(3), np.allclose(matrix, loaded_npy))

## Pandas: индексация, concat, melt, join и импорт/экспорт

In [ ]:
content = load_content_frame()
first_rows = content.iloc[:3, :6]             # позиционная индексация
tiktok_rows = content.loc[content['platform'].eq('tiktok'), ['author', 'views', 'category']]
combined = pd.concat([first_rows, first_rows], ignore_index=True)
long_metrics = content.melt(id_vars=['external_id', 'platform'], value_vars=['views', 'likes', 'comments'], var_name='metric', value_name='value')
platform_dim = pd.DataFrame({'platform': ['tiktok', 'telegram', 'youtube', 'missing'], 'label': ['Short video', 'Community', 'Video', 'No content']})
inner = content.merge(platform_dim, on='platform', how='inner')
left = platform_dim.merge(content[['platform', 'external_id']], on='platform', how='left')
right = content[['platform', 'external_id']].merge(platform_dim, on='platform', how='right')
outer = content[['platform', 'external_id']].merge(platform_dim, on='platform', how='outer')
csv_path = ARTIFACTS / 'notebook_content.csv'
content.to_csv(csv_path, index=False)          # export
imported = pd.read_csv(csv_path)                # import
print(len(imported), len(combined), len(long_metrics), [len(x) for x in (inner, left, right, outer)])

## Matplotlib: histogram, heatmap, subplot, 3D и SVG

In [ ]:
fig = plt.figure(figsize=(14, 8))
ax_hist = fig.add_subplot(2, 2, 1)
ax_hist.hist(content['views'], bins=6, color='#2563eb', edgecolor='white')
ax_hist.set(title='Histogram просмотров', xlabel='Views')

ax_heat = fig.add_subplot(2, 2, 2)
corr = content[['views', 'likes', 'comments', 'shares']].corr()
heatmap = ax_heat.imshow(corr, cmap='viridis', vmin=-1, vmax=1)
ax_heat.set(title='Heatmap корреляций', xticks=range(4), yticks=range(4), xticklabels=corr.columns, yticklabels=corr.columns)
fig.colorbar(heatmap, ax=ax_heat)

ax_line = fig.add_subplot(2, 2, 3)
ax_line.plot(content['created_at'], content['trend_score'], marker='o')
ax_line.tick_params(axis='x', rotation=35)
ax_line.set(title='Subplot trend score', ylabel='Score')

ax_3d = fig.add_subplot(2, 2, 4, projection='3d')
ax_3d.scatter(content['views'], content['likes'], content['trend_score'], c=content['trend_score'], cmap='plasma')
ax_3d.set(xlabel='Views', ylabel='Likes', zlabel='Trend score', title='3D scatter')
fig.tight_layout()
fig.savefig(ARTIFACTS / 'data_stack_demo.svg', format='svg')
plt.show()